# Phase 15: PF-X — self-contained particle filter (local research)

Local research notebook, same conventions as notebooks 01–14: lives in
`notebooks/`, reads `../data/raw/...` directly, no environment variables, no
Kaggle logic. Its sibling `rogii-wellbore-submission-pfx.ipynb` carries the
**byte-identical engine** with plain Kaggle paths.

Mechanisms (knowledge carried over; artifacts dropped):
- **likelihood-weighted seed ensembling** (we previously only averaged seeds),
- **soft dZ-velocity likelihood** (the falsified version was a hard pull — this
  evidence form was never tested),
- **centered both-sides GR mixing** (GR is fully observed on eval rows; our
  earlier falsification only tested causal smoothing),
- **projection post-processing**, measured as a lever, not assumed.

Dev-well evidence (real well, floor 7.29): base 11.63 → +both soft likelihoods
8.33 → **+likelihood-weighted seeds 4.21**. One well proves mechanics only:
the fleet CV + ablation below decide.

## Config

In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
from pathlib import Path
import numpy as np, pandas as pd

TRAIN_DIR = Path("../data/raw/train")
TEST_DIR = Path("../data/raw/test")     # used only by the optional sanity cell

WELL_SAMPLE = 250        # None = all train wells
ABLATION_WELLS = 60
EVAL_FRAC = 0.73

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

print(f"train dir exists: {TRAIN_DIR.exists()} | test dir exists: {TEST_DIR.exists()}")

train dir exists: True | test dir exists: True


## Engine (byte-identical copy lives in the submission notebook)

In [2]:
def load_raw_well(hz_path, tw_path):
    h = pd.read_csv(hz_path).sort_values("MD").reset_index(drop=True)
    t = pd.read_csv(tw_path)
    tw = t.dropna(subset=["TVT", "GR"]).sort_values("TVT")
    twt = tw["TVT"].values.astype(float)
    twg = tw["GR"].clip(0, 300).values.astype(float)
    Z = h["Z"].values.astype(float); MD = h["MD"].values.astype(float)
    gr = pd.Series(h["GR"].clip(0, 300).values).interpolate(
        limit_direction="both").fillna(90.0).values
    return h, twt, twg, Z, MD, gr

def pfx(tvt_known, Z, MD, gr, twt, twg, kn, ev,
        S=16, N=500, mom=0.998, vn=0.002, pn=0.005,
        dz_lik=True, zsig_mult=2.0, gr_wt=0.3, sm_win=5,
        lik_weight=True, temp=1.0, seed=42):
    # tvt_known: full-length array, finite (trusted) on kn; ev values are ignored.
    rng = np.random.default_rng(seed)
    last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt_known[kn], twt, twg)), 10.0, 60.0))
    dzk = np.diff(Z[kn]); dvt = np.diff(tvt_known[kn]); dmk = np.diff(MD[kn]); m = dmk > 0
    if m.sum() >= 10:
        vz = dzk[m] / dmk[m]; vt = dvt[m] / dmk[m]
        A = np.column_stack([vz, np.ones_like(vz)])
        c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt = float(c[0]), float(c[1])
        zsig = max(float(np.std(vt - (c[0] * vz + c[1]))), 1e-3)
    else:
        beta, icpt, zsig = -1.0, 0.0, 0.1
    tl = kn[-20:]; d2 = np.diff(tvt_known[tl]); dm2 = np.diff(MD[tl]); mm = dm2 > 0
    iv = float(np.median(d2[mm] / dm2[mm])) if mm.sum() >= 3 else 0.0
    gr_s = pd.Series(gr).rolling(sm_win, center=True, min_periods=1).mean().values
    twg_s = pd.Series(twg).rolling(sm_win, center=True, min_periods=1).mean().values
    lo, hi = twt[0] - 50.0, twt[-1] + 50.0
    pos = tvt_known[last] + 0.5 * rng.standard_normal((S, N))
    vel = iv + 0.02 * rng.standard_normal((S, N))
    w = np.ones((S, N)) / N
    LL = np.zeros(S)
    out = np.empty((S, len(ev)))
    prev_md, prev_z = MD[last], Z[last]
    for i, idx in enumerate(ev):
        dm = max(MD[idx] - prev_md, 1.0); dzd = (Z[idx] - prev_z) / dm
        vel = mom * vel + vn * rng.standard_normal((S, N))
        pos = np.clip(pos + vel * dm + pn * rng.standard_normal((S, N)), lo, hi)
        g = gr[idx]
        if np.isfinite(g):
            eg = np.interp(pos.ravel(), twt, twg).reshape(S, N)
            d2r = ((g - eg) / gs) ** 2
            if gr_wt > 0:
                eg2 = np.interp(pos.ravel(), twt, twg_s).reshape(S, N)
                d2s = ((gr_s[idx] - eg2) / (gs * 1.5)) ** 2
                lk = (1 - gr_wt) * np.exp(-0.5 * np.minimum(d2r, 600.0)) \
                     + gr_wt * np.exp(-0.5 * np.minimum(d2s, 600.0))
            else:
                lk = np.exp(-0.5 * np.minimum(d2r, 600.0))
            w = w * np.maximum(lk, 1e-300)
        if dz_lik:
            ve = beta * dzd + icpt
            dv = (vel - ve) / max(zsig * zsig_mult, 5e-3)
            w = w * np.maximum(np.exp(-0.5 * np.minimum(dv * dv, 600.0)), 1e-300)
        ws = w.sum(1, keepdims=True)
        LL += np.log(np.maximum(ws.ravel() / N, 1e-300))
        w = np.where(ws > 0, w / np.maximum(ws, 1e-300), 1.0 / N)
        ess = 1.0 / np.maximum((w * w).sum(1), 1e-300)
        for s in np.where(ess < 0.5 * N)[0]:
            ci = np.clip(np.searchsorted(np.cumsum(w[s]),
                 (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1)
            pos[s] = pos[s][ci] + 0.1 * rng.standard_normal(N)
            vel[s] = vel[s][ci] + 0.003 * rng.standard_normal(N)
            w[s] = 1.0 / N
        out[:, i] = (w * pos).sum(1)
        prev_md, prev_z = MD[idx], Z[idx]
    if lik_weight and S > 1:
        LLn = (LL - LL.max()) / max(temp, 1e-6)
        sw = np.exp(LLn); sw /= sw.sum()
        return (out * sw[:, None]).sum(0)
    return out.mean(0)

def project(pred, Z_ev, MD_ev, anchor_tvt, anchor_z, anchor_md, deg=4, blend=0.75):
    u = pred + Z_ev - (anchor_tvt + anchor_z)
    span = MD_ev[-1] - anchor_md
    if len(pred) < deg + 3 or span <= 0:
        return pred
    s = (MD_ev - anchor_md) / span
    c = np.polyfit(s, u, deg)
    for _ in range(4):
        r = u - np.polyval(c, s)
        sc = np.median(np.abs(r)) * 1.4826 + 1e-6
        c = np.polyfit(s, u, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    u2 = blend * np.polyval(c, s) + (1 - blend) * u
    return u2 + (anchor_tvt + anchor_z) - Z_ev

## Fleet CV (the decider)

73% tail mask per well. References: PF v4 was 12.76 pooled CV → 11.375 LB;
current best LB 10.049. CV→LB has transferred at-or-better all project.

In [3]:
files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
wids = [f.name.split("__")[0] for f in files
        if (TRAIN_DIR / f"{f.name.split('__')[0]}__typewell.csv").exists()]
if WELL_SAMPLE and WELL_SAMPLE < len(wids):
    wids = sorted(np.random.default_rng(123).choice(wids, WELL_SAMPLE, replace=False))
print(f"CV on {len(wids)} wells")

rows = []; t0 = time.time()
for k, wid in enumerate(wids):
    try:
        h, twt, twg, Z, MD, gr = load_raw_well(
            TRAIN_DIR / f"{wid}__horizontal_well.csv",
            TRAIN_DIR / f"{wid}__typewell.csv")
    except Exception:
        continue
    if "TVT" not in h or h["TVT"].isna().all() or len(twt) < 3:
        continue
    tvt = h["TVT"].values.astype(float); n = len(h)
    kcut = int(round(n * EVAL_FRAC))
    if kcut < 20 or n - kcut < 20:
        continue
    kn = np.arange(0, n - kcut); ev = np.arange(n - kcut, n)
    true = tvt[ev]
    p = pfx(tvt, Z, MD, gr, twt, twg, kn, ev)
    pj = project(p, Z[ev], MD[ev], tvt[kn[-1]], Z[kn[-1]], MD[kn[-1]])
    rows.append(dict(well=wid, n_eval=len(ev),
                     floor=rmse(tvt[kn[-1]], true),
                     pfx=rmse(p, true), pfx_proj=rmse(pj, true)))
    if (k + 1) % 25 == 0:
        print(f"  ...{k+1}/{len(wids)} [{(time.time()-t0)/60:.1f} min]")

cv = pd.DataFrame(rows)
cv.to_csv("../data/interim/pfx_cv.csv", index=False)

def pooled(col):
    return float(np.sqrt((cv[col] ** 2 * cv["n_eval"]).sum() / cv["n_eval"].sum()))
print(f"\n{'model':12s}{'pooled':>9s}{'per-well':>10s}")
for c in ["floor", "pfx", "pfx_proj"]:
    print(f"{c:12s}{pooled(c):9.3f}{cv[c].mean():10.3f}")
print("(references: PF v4 12.76 pooled CV -> 11.375 LB | best LB 10.049)")
print("PROJECTION VERDICT:",
      "helps -> set USE_PROJECTION=True in the submission notebook"
      if cv["pfx_proj"].mean() < cv["pfx"].mean() else
      "does not help -> leave USE_PROJECTION=False")

CV on 250 wells
  ...25/250 [1.3 min]
  ...50/250 [2.6 min]
  ...75/250 [3.9 min]
  ...100/250 [5.3 min]
  ...125/250 [6.6 min]
  ...150/250 [7.8 min]
  ...175/250 [9.2 min]
  ...200/250 [10.5 min]
  ...225/250 [11.8 min]
  ...250/250 [13.1 min]

model          pooled  per-well
floor          18.009    13.873
pfx            15.172    11.525
pfx_proj       15.020    11.298
(references: PF v4 12.76 pooled CV -> 11.375 LB | best LB 10.049)
PROJECTION VERDICT: helps -> set USE_PROJECTION=True in the submission notebook


## Mechanism ablation (S=8 quick runs)

The dev-well ordering (base < +dz_lik < +both < +lik_weight) must reproduce at
scale to be believed. Any mechanism that fails here gets its default flipped
off in `pfx(...)` — in both notebooks.

In [4]:
ab_wids = wids[:min(ABLATION_WELLS, len(wids))]
CFGS = {
    "base_uniform":  dict(dz_lik=False, gr_wt=0.0, lik_weight=False),
    "+dz_lik":       dict(dz_lik=True,  gr_wt=0.0, lik_weight=False),
    "+gr_centered":  dict(dz_lik=False, gr_wt=0.3, lik_weight=False),
    "+both":         dict(dz_lik=True,  gr_wt=0.3, lik_weight=False),
    "+lik_weight":   dict(dz_lik=True,  gr_wt=0.3, lik_weight=True),
}
res = {k: [] for k in CFGS}
t0 = time.time()
for wid in ab_wids:
    try:
        h, twt, twg, Z, MD, gr = load_raw_well(
            TRAIN_DIR / f"{wid}__horizontal_well.csv",
            TRAIN_DIR / f"{wid}__typewell.csv")
    except Exception:
        continue
    if "TVT" not in h or h["TVT"].isna().all() or len(twt) < 3:
        continue
    tvt = h["TVT"].values.astype(float); n = len(h)
    kcut = int(round(n * EVAL_FRAC))
    if kcut < 20 or n - kcut < 20:
        continue
    kn = np.arange(0, n - kcut); ev = np.arange(n - kcut, n)
    for name, kw in CFGS.items():
        p = pfx(tvt, Z, MD, gr, twt, twg, kn, ev, S=8, **kw)
        res[name].append(rmse(p, tvt[ev]))
print(f"ablation on {len(res['base_uniform'])} wells [{(time.time()-t0)/60:.1f} min]:")
for name in CFGS:
    print(f"  {name:14s} per-well {np.mean(res[name]):7.3f}")

ablation on 60 wells [7.4 min]:
  base_uniform   per-well  11.729
  +dz_lik        per-well  11.018
  +gr_centered   per-well  12.827
  +both          per-well  11.444
  +lik_weight    per-well  10.683


## Optional: local sanity check on `../data/raw/test`

Runs the engine on the three local test wells and reports predicted ranges vs
known ranges. Writes nothing — submissions come from the sibling notebook.

In [5]:
if TEST_DIR.exists():
    for p in sorted(TEST_DIR.glob("*__horizontal_well.csv")):
        wid = p.name.replace("__horizontal_well.csv", "")
        tp = TEST_DIR / f"{wid}__typewell.csv"
        if not tp.exists():
            print(f"{wid}: no typewell"); continue
        h, twt, twg, Z, MD, gr = load_raw_well(p, tp)
        ti = h["TVT_input"].values.astype(float)
        kn = np.where(np.isfinite(ti))[0]; ev = np.where(~np.isfinite(ti))[0]
        if len(ev) == 0 or len(kn) < 20:
            print(f"{wid}: nothing to predict"); continue
        tvt_known = ti.copy(); tvt_known[ev] = ti[kn[-1]]
        pred = pfx(tvt_known, Z, MD, gr, twt, twg, kn, ev)
        print(f"{wid}: eval={len(ev)}  pred [{pred.min():.1f}, {pred.max():.1f}]  "
              f"known [{np.nanmin(ti):.1f}, {np.nanmax(ti):.1f}]")
else:
    print("no local test dir — skipping")

000d7d20: eval=3836  pred [11736.1, 11753.7]  known [11236.0, 11756.1]
00bbac68: eval=6014  pred [12223.6, 12241.7]  known [11406.6, 12225.0]
00e12e8b: eval=4301  pred [11596.8, 11615.2]  known [10606.2, 11604.8]


## Reading

1. **CV table**: `pfx` pooled below ~11 → a dependency-free single engine is
   competitive with the 10.049 blend; submit via the sibling notebook.
2. **Projection verdict** line → sets `USE_PROJECTION` in the submission notebook.
3. **Ablation ordering** → any failing mechanism gets turned off in both files.